In [ ]:
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [ ]:
path_to_figures = "/path/to/Figures"
folder_figures = "BP_analysis"

import os
if not os.path.exists(os.path.join(path_to_figures, folder_figures)):
    os.makedirs(os.path.join(path_to_figures, folder_figures))

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Define a diverging colormap from pink to cyan
colors = ["navy", "#f7f7f7", "tomato"]  # Pink -> White -> Cyan
cmap = LinearSegmentedColormap.from_list("tomatoNavy", colors)


In [ ]:
metadata_path = "path/to/AstroBF_fastq/RN22346/RN22346.xlsx"
metadata = pd.read_excel(metadata_path)
metadata['category'] = metadata["Sample Genotype"] + "_" + metadata["Sample Treatment"]

In [ ]:
path_analysis_per_patient = "path/to/BP_analysis_per_patient/GO"


In [ ]:
csv_files = glob.glob(path_analysis_per_patient + '/*.csv')


In [ ]:
samples_names = []
dataframes = []
increased = []
decreased = []
for file in csv_files:
    # Extract the sample name from the file path
    sample_name = file.split('/')[-1].split('.csv')[0]
    samples_names.append(sample_name)
    df = pd.read_csv(file)
    df.replace('NA', np.NaN)
    df.dropna(inplace=True)
    increase = df[(df[f'pvalue_{sample_name}'] < 0.05) & (df[f'gene_in_GO_{sample_name}'] == 'increase')].sort_values(f'pvalue_{sample_name}')
    increased.append(increase)
    decrease = df[(df[f'pvalue_{sample_name}'] < 0.05) & (df[f'gene_in_GO_{sample_name}'] == 'decrease')].sort_values(f'pvalue_{sample_name}')
    decreased.append(decrease)
    dataframes.append(df)


In [ ]:
df_pvalues = pd.DataFrame()
for df, sample_name in zip(dataframes, samples_names):
    sign_ = (df[f'gene_in_GO_{sample_name}'] == 'increase').astype(int).replace(0, -1)
    pval_log = -np.log10(df[f'pvalue_{sample_name}'])
    df_pvalues[sample_name] = sign_*pval_log
df_pvalues.index = df['Description'].values

In [ ]:
df_pvalues

In [ ]:
df_pvalues

In [ ]:
treatments = {'ctrl2_basal': ['TAH5505A1'], 
              'mut2_basal': ['TAH5505A10'], 
              'mut2_A1 low': ['TAH5505A11'], 
              'mut2_A1 high': ['TAH5505A12'], 
              'mut4_basal': ['TAH5505A13'], 
              'mut4_A1 low': ['TAH5505A14'], 
              'mut4_A1 high': ['TAH5505A15'], 
              'ctrl6_basal': ['TAH5505A16'], 
              'ctrl6_A1 low': ['TAH5505A17'], 
              'ctrl6_A1 high': ['TAH5505A18'], 
              'ctrl2_A1 low': ['TAH5505A2'], 
              'ctrl2_A1 high': ['TAH5505A3'], 
              'ctrl5_basal': ['TAH5505A4'], 
              'ctrl5_A1 low': ['TAH5505A5'], 
              'ctrl5_A1 high': ['TAH5505A6'], 
              'mut1_basal': ['TAH5505A7'], 
              'mut1_A1 low': ['TAH5505A8'], 
              'mut1_A1 high': ['TAH5505A9']}

treatments_cell_lines = {'ctrl_basal': ['TAH5505A1', 'TAH5505A16', 'TAH5505A4'],
 'ctrl_A1 low': ['TAH5505A17', 'TAH5505A2', 'TAH5505A5'],
 'ctrl_A1 high': ['TAH5505A18', 'TAH5505A3', 'TAH5505A6'],
 'vcp_basal': ['TAH5505A10', 'TAH5505A13', 'TAH5505A7'],
 'vcp_A1 low': ['TAH5505A11', 'TAH5505A14', 'TAH5505A8'],
 'vcp_A1 high': ['TAH5505A12', 'TAH5505A15', 'TAH5505A9']}

palette_genotype = {"ctrl_basal": 'palegreen', "ctrl_A1 low": "limegreen", "ctrl_A1 high": "forestgreen", "vcp_basal": "pink", "vcp_A1 low": "hotpink", "vcp_A1 high": "deeppink" }
palette_treatments = {"ctrl_basal": 'dimgray', "ctrl_A1 low": "deepskyblue", "ctrl_A1 high": "darkorange", "vcp_basal": "dimgray", "vcp_A1 low": "deepskyblue", "vcp_A1 high": "darkorange" }

In [ ]:
for condition, samples in treatments_cell_lines.items():
    df_pvalues[condition] = df_pvalues[samples].mean(axis=1)
    df_pvalues.drop(samples, axis=1, inplace=True)

In [ ]:
df_pvalues

In [ ]:
# More robust cleaning of infinite and NaN values
print("Data shape before cleaning:", df_pvalues.shape)
print("Number of infinite values:", np.isinf(df_pvalues).sum().sum())
print("Number of NaN values:", df_pvalues.isna().sum().sum())

# Replace infinite values with the maximum finite value
max_finite_value = df_pvalues.replace([np.inf, -np.inf], np.nan).max().max()
df_pvalues = df_pvalues.replace([np.inf, -np.inf], max_finite_value)

# Fill any remaining NaN values with 0
df_pvalues = df_pvalues.fillna(0)

print("After cleaning:")
print("Number of infinite values:", np.isinf(df_pvalues).sum().sum())
print("Number of NaN values:", df_pvalues.isna().sum().sum())
print("Data range:", df_pvalues.min().min(), "to", df_pvalues.max().max())

In [ ]:
df_filtered = df_pvalues[(df_pvalues.abs() > -np.log(0.05)).any(axis=1)]   


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt



# Additional data cleaning for clustering
print("Before additional cleaning:")
print("Data shape:", df_filtered.shape)
print("Data range:", df_filtered.min().min(), "to", df_filtered.max().max())

# Remove rows with zero variance (all identical values)
row_variance = df_filtered.var(axis=1)
print(f"Rows with zero variance: {(row_variance == 0).sum()}")
df_clean = df_filtered[row_variance > 0]

# Remove columns with zero variance  
col_variance = df_clean.var(axis=0)
print(f"Columns with zero variance: {(col_variance == 0).sum()}")
df_clean = df_clean.loc[:, col_variance > 0]

col_colors_clean = [palette_treatments[condition] for condition in df_clean.columns]

print("After cleaning:")
print("Data shape:", df_clean.shape)
print("Data range:", df_clean.min().min(), "to", df_clean.max().max())

# Verify that we have finite values for clustering
print("Final check - any infinite or NaN values:", 
      np.isinf(df_clean).any().any() or df_clean.isna().any().any())

# Create clustermap with multiple fallback options


In [ ]:
col_colors_clean = [palette_genotype[condition] for condition in df_clean.columns]

# Create clustermap with multiple fallback options

# First try with standard scaling
g = sns.clustermap(df_clean, 
                    col_colors=col_colors_clean,
                    col_cluster=True, 
                    row_cluster=True, 
                    yticklabels=False, 
                    cmap=cmap, 
                    center=0,
                    figsize=(5, 10))

g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(), 
    rotation=45, 
    ha='right'
)
plt.savefig(os.path.join(path_to_figures, folder_figures, "BP_clustermap_avg_per_condition.pdf"), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
col_colors_clean = [palette_treatments[condition] for condition in df_clean.columns]

# Create clustermap with multiple fallback options

# First try with standard scaling
g = sns.clustermap(df_clean, 
                    col_colors=col_colors_clean,
                    col_cluster=True, 
                    row_cluster=True, 
                    yticklabels=False, 
                    cmap=cmap, 
                    center=0,
                    figsize=(5, 10))

g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(), 
    rotation=45, 
    ha='right'
)
plt.savefig(os.path.join(path_to_figures, folder_figures, "BP_clustermap_avg_per_condition_treatments.pdf"), bbox_inches='tight', dpi=300)

In [ ]:
treatments = {k: [k] for k in df_pvalues.columns}

In [ ]:
from melapa.OmicsDataset import OmicsDataset

pvalues_dataset = OmicsDataset(df=df_pvalues, 
                               treatments=treatments, 
                               nb_comp=6, 
                               df_log=df_pvalues)

In [ ]:
df_pvalues

In [ ]:
palette = [palette_treatments[condition] for condition in df_pvalues.columns]

In [ ]:
pvalues_dataset.hierarchical_clustering(palette=palette, treatments=treatments, color_threshold=0)
plt.savefig(os.path.join(path_to_figures, folder_figures, "hierarchical_clustering_all_BP_avg_per_condition_treatments.pdf"), bbox_inches='tight', dpi=300)

In [ ]:
pvalues_dataset.perform_svd_analysis(top_n=6)
plt.savefig(os.path.join(path_to_figures, folder_figures, "scree_plot_first_comp_removed.pdf"), bbox_inches='tight', dpi=300)

In [ ]:
explained_variance_w1 = pvalues_dataset.s_w1*100 / pvalues_dataset.s_w1.sum()

In [ ]:
df_var = pd.DataFrame({'Explained Variance (%)': explained_variance_w1}, index=[f'Component {i+1}' for i in range(len(explained_variance_w1))])

In [ ]:
sns.barplot(df_var, x=df_var.index, y="Explained Variance (%)", palette='viridis')
sns.despine()
plt.savefig(os.path.join(path_to_figures, folder_figures, "explained_variance_barplot.pdf"), bbox_inches='tight', dpi=300)

In [ ]:
pvalues_dataset.pca['Sample Treatment'] = pvalues_dataset.pca['treatment'].apply(lambda x: x.split('_')[1])
pvalues_dataset.pca['Sample Genotype'] = pvalues_dataset.pca['treatment'].apply(lambda x: x.split('_')[0])
pvalues_dataset.pca['mutation'] = pvalues_dataset.pca['treatment'].apply(lambda x: x.split('_')[0])



In [ ]:
pvalues_dataset.pca

In [ ]:
palette_genotype = {"ctrl": 'palegreen', "vcp": "hotpink"}
plt.figure(figsize=(20, 15))
for pc in range(6):
    plt.subplot(5, 4, pc+1)
    for genotype in pvalues_dataset.pca['Sample Genotype'].unique():
        subset = pvalues_dataset.pca[pvalues_dataset.pca['Sample Genotype'] == genotype]
        # subset.index = [idx.split('_')[1] for idx in subset.index]
        subset.set_index("Sample Treatment", inplace=True)
        subset = subset.loc[["basal", "A1 low", "A1 high"]]
        plt.scatter(x=subset.index, y=subset[f'PC{pc+1}'], label=genotype, color=palette_genotype[genotype])
        plt.plot(subset.index, subset[f'PC{pc+1}'], color=palette_genotype[genotype])
    if pc == 3:
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    sns.despine()
    plt.title(f"PC{pc+1}", weight='bold')
plt.tight_layout()
plt.savefig(os.path.join(path_to_figures, folder_figures, "boxplot_pca_per_mutation.pdf"), bbox_inches='tight', dpi=300)

In [ ]:
from melapa.helper_plot import boxplot_pca_significance

boxplot_pca_significance(pvalues_dataset.pca, treatment_col='mutation', fontsize=10, palette={'ctrl': 'limegreen', 'vcp': 'hotpink'})
plt.savefig(os.path.join(path_to_figures, folder_figures, "boxplot_pca_per_mutation.pdf"), bbox_inches='tight', dpi=300)

In [ ]:
pvalues_dataset.pca['treatment'] = pvalues_dataset.pca['Sample Treatment']

In [ ]:
%%capture cap --no-display

from melapa.LMM_helper import lmer_per_PC_re

## See which component captures the difference sensitive/resistant

plt.figure(figsize=(10, 5))
## See which component captures the batch
anovas = []
p_vals = []

for i in range(6):
    PC = i+1
    anova, p_val = lmer_per_PC_re(pvalues_dataset.pca, random_effects=['treatment'], fixed_effect='mutation', PC=PC)
    anovas.append(anova)
    p_vals.append(p_val)

plt.subplot(1, 2, 1)
sns.barplot(x=np.arange(1, 7), y=-np.log10(p_vals), color='grey')
plt.axhline(y=-np.log10(0.05), color='red', linestyle='--')
plt.xticks(rotation=90, fontsize=8);
plt.xlabel('PCs')
plt.ylabel('-log10(p-value)')
plt.title(f'Mutation \n ANOVA between full and reduced model \n p-value')

delta_deviances = []
for PC in range(len(anovas)):
    delta_deviance = anovas[PC].loc['MODEL2', 'deviance'] - anovas[PC].loc['MODEL1', 'deviance'] ## Difference between the full and reduced model
    delta_deviances.append(delta_deviance)

plt.subplot(1, 2, 2)
sns.barplot(x=[f"PC{i+1}" for i in range(len(anovas))], y=delta_deviances, color='grey')
#plt.axhline(y=-np.log10(0.05), color='red', linestyle='--')
plt.xticks(rotation=90, fontsize=8);
plt.xlabel('PCs')
plt.ylabel(r'$\Delta$ deviance')
plt.title(f'Mutation \n ANOVA between full and reduced model \n Deviance')

plt.tight_layout()
plt.savefig(os.path.join(path_to_figures, folder_figures, "lmm_mutation_per_PC.pdf"), bbox_inches='tight', dpi=300)

In [ ]:
PCs = [f"PC{PC+1}" for PC in list(np.argwhere(np.array(p_vals) < 0.05).flatten())] + ['mutation']
sns.pairplot(pvalues_dataset.pca[PCs], hue='mutation',  palette={'ctrl': 'limegreen', 'vcp': 'hotpink'})
plt.savefig(os.path.join(path_to_figures, folder_figures, "pairplot_significant_pcs_mutation.pdf"), bbox_inches='tight', dpi=300)

In [ ]:
boxplot_pca_significance(pvalues_dataset.pca, treatment_col='treatment', fontsize=10, palette={'basal': 'dimgray', 'A1 low': 'deepskyblue', "A1 high": "darkorange"})
plt.savefig(os.path.join(path_to_figures, folder_figures, "boxplot_pca_per_treatment.pdf"), bbox_inches='tight', dpi=300)

In [ ]:
%%capture cap --no-display

from melapa.LMM_helper import lmer_per_PC_re

## See which component captures the difference sensitive/resistant

plt.figure(figsize=(10, 5))
## See which component captures the batch
anovas = []
p_vals = []

for i in range(6):
    PC = i+1
    anova, p_val = lmer_per_PC_re(pvalues_dataset.pca, random_effects=['mutation'], fixed_effect='treatment', PC=PC)
    anovas.append(anova)
    p_vals.append(p_val)

plt.subplot(1, 2, 1)
sns.barplot(x=np.arange(1, 7), y=-np.log10(p_vals), color='grey')
plt.axhline(y=-np.log10(0.05), color='red', linestyle='--')
plt.xticks(rotation=90, fontsize=8);
plt.xlabel('PCs')
plt.ylabel('-log10(p-value)')
plt.title(f'Treatment \n ANOVA between full and reduced model \n p-value')

delta_deviances = []
for PC in range(len(anovas)):
    delta_deviance = anovas[PC].loc['MODEL2', 'deviance'] - anovas[PC].loc['MODEL1', 'deviance'] ## Difference between the full and reduced model
    delta_deviances.append(delta_deviance)

plt.subplot(1, 2, 2)
sns.barplot(x=[f"PC{i+1}" for i in range(len(anovas))], y=delta_deviances, color='grey')
#plt.axhline(y=-np.log10(0.05), color='red', linestyle='--')
plt.xticks(rotation=90, fontsize=8);
plt.xlabel('PCs')
plt.ylabel(r'$\Delta$ deviance')
plt.title(f'Treatment \n ANOVA between full and reduced model \n Deviance')

plt.tight_layout()
plt.savefig(os.path.join(path_to_figures, folder_figures, "lmm_treatment_per_PC.pdf"), bbox_inches='tight', dpi=300)

In [ ]:
PCs = [f"PC{PC+1}" for PC in list(np.argwhere(np.array(p_vals) < 0.05).flatten())] + ['treatment']
sns.pairplot(pvalues_dataset.pca[PCs], hue='treatment',  palette={'basal': 'dimgray', 'A1 low': 'deepskyblue', "A1 high": "darkorange"})
plt.savefig(os.path.join(path_to_figures, folder_figures, "pairplot_significant_pcs_treatment.pdf"), bbox_inches='tight', dpi=300)

In [ ]:
projection_PC1 = pvalues_dataset.Vt_w1[0]
indexes_top = np.argsort(projection_PC1)[-50:]
indexes_bottom = np.argsort(projection_PC1)[:50]

top_highest_PC1 = list(pvalues_dataset.df.reset_index().iloc[indexes_top]['index'].values)
top_lowest_PC1 = list(pvalues_dataset.df.reset_index().iloc[indexes_bottom]['index'].values)



In [ ]:
top_proj40 = pd.concat([
              pvalues_dataset.X_without_PC1.T.loc[top_highest_PC1],
              pvalues_dataset.X_without_PC1.T.loc[top_lowest_PC1],
              ]).drop_duplicates(keep='first')


# top_proj40 = top_proj40[ordered_treatments]

In [ ]:
# Update col_colors to match remaining columns
col_colors_clean = [palette_treatments[treatment] for treatment in top_proj40.columns]

print("After cleaning:")
print("Data shape:", top_proj40.shape)
print("Data range:", top_proj40.min().min(), "to", top_proj40.max().max())

# Verify that we have finite values for clustering
print("Final check - any infinite or NaN values:", 
      np.isinf(top_proj40).any().any() or top_proj40.isna().any().any())

# Create clustermap with multiple fallback options

# First try with standard scaling
g = sns.clustermap(top_proj40, 
                    col_colors=col_colors_clean,
                    col_cluster=True, 
                    row_cluster=True, 
                    yticklabels=True, 
                    cmap=cmap, 
                    center=0,
                    figsize=(12, 20))
g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(), 
    rotation=45, 
    ha='right'
)
plt.savefig(os.path.join(path_to_figures, folder_figures, "BP_clustermap_top50_PC1_removed.pdf"), bbox_inches='tight', dpi=300)

In [ ]:
# Convert the CSV data into a dictionary and DataFrame
import io

go_terms_csv = """Category,GO_Term
Cytoskeleton_Adhesion_Motility,Myosin II complex
Cytoskeleton_Adhesion_Motility,Cell cortex
Cytoskeleton_Adhesion_Motility,Myofibril
Cytoskeleton_Adhesion_Motility,Filamentous actin
Cytoskeleton_Adhesion_Motility,Actin filament
Cytoskeleton_Adhesion_Motility,Z disc
Cytoskeleton_Adhesion_Motility,Sarcomere organization
Cytoskeleton_Adhesion_Motility,Filopodium
Cytoskeleton_Adhesion_Motility,Cell projection
Cytoskeleton_Adhesion_Motility,Regulation of focal adhesion assembly
Cytoskeleton_Adhesion_Motility,Adherens junction
Cytoskeleton_Adhesion_Motility,Tight junction
Cytoskeleton_Adhesion_Motility,Cell-cell junction
Cytoskeleton_Adhesion_Motility,Cell-cell junction organization
Cytoskeleton_Adhesion_Motility,Cell motility
Cytoskeleton_Adhesion_Motility,Cell migration
Cytoskeleton_Adhesion_Motility,Leukocyte migration
Cytoskeleton_Adhesion_Motility,Neuron migration
Cytoskeleton_Adhesion_Motility,Epithelial to mesenchymal transition
Cytoskeleton_Adhesion_Motility,Regulation of cell morphogenesis
Cytoskeleton_Adhesion_Motility,Negative regulation of fibroblast proliferation
Cytoskeleton_Adhesion_Motility,Negative regulation of vascular associated smooth muscle cell proliferation
Cytoskeleton_Adhesion_Motility,Negative regulation of cell population proliferation
Cytoskeleton_Adhesion_Motility,Positive regulation of cell differentiation
Cytoskeleton_Adhesion_Motility,Positive regulation of myoblast differentiation
ECM_Tissue_Remodeling,Hyaluronic acid binding
ECM_Tissue_Remodeling,Collagen fibril organization
ECM_Tissue_Remodeling,Extracellular matrix organization
ECM_Tissue_Remodeling,Extracellular matrix binding
ECM_Tissue_Remodeling,Wound healing
ECM_Tissue_Remodeling,Response to wounding
ECM_Tissue_Remodeling,Heart morphogenesis
ECM_Tissue_Remodeling,Blood circulation
ECM_Tissue_Remodeling,Megakaryocyte development
Neuronal_Synaptic,Synapse
Neuronal_Synaptic,Synapse organization
Neuronal_Synaptic,Schaffer collateral - CA1 synapse
Neuronal_Synaptic,Long-term synaptic potentiation
Neuronal_Synaptic,Visual learning
Neuronal_Synaptic,Response to activity
Neuronal_Synaptic,Response to electrical stimulus
Neuronal_Synaptic,Neuron apoptotic process
Neuronal_Synaptic,Response to axon injury
Neuronal_Synaptic,Cell body
Neuronal_Synaptic,Semaphorin-plexin signaling pathway
Immune_Antiviral,Defense response
Immune_Antiviral,Defense response to virus
Immune_Antiviral,Antiviral innate immune response
Immune_Antiviral,Innate immune response
Immune_Antiviral,Immune response
Immune_Antiviral,Cellular response to virus
Immune_Antiviral,Negative regulation of viral genome replication
Immune_Antiviral,Negative regulation of viral entry into host cell
Immune_Antiviral,Type I interferon-mediated signaling pathway
Immune_Antiviral,Positive regulation of interferon-alpha production
Immune_Antiviral,Positive regulation of interferon-beta production
Immune_Antiviral,Positive regulation of T cell mediated cytotoxicity
Immune_Antiviral,Leukocyte cell-cell adhesion
Immune_Antiviral,Positive regulation of monocyte chemotaxis
Immune_Antiviral,Chemokine activity
Immune_Antiviral,Chemoattractant activity
Immune_Antiviral,Antibacterial humoral response
Immune_Antiviral,Peptide antigen binding
Immune_Antiviral,Phagocytic vesicle membrane
Immune_Antiviral,Tertiary granule lumen
Immune_Antiviral,Superoxide anion generation
Immune_Antiviral,Positive regulation of nitric oxide biosynthetic process
Intracellular_Organization,Golgi lumen
Intracellular_Organization,Endoplasmic reticulum
Intracellular_Organization,Lumenal side of endoplasmic reticulum membrane
Intracellular_Organization,ER to Golgi transport vesicle membrane
Intracellular_Organization,Recycling endosome membrane
Intracellular_Organization,Cytosolic small ribosomal subunit
Intracellular_Organization,Mitotic spindle assembly
Intracellular_Organization,SWI/SNF complex
Intracellular_Organization,Regulation of protein-containing complex assembly
Intracellular_Organization,Protein homotetramerization
Intracellular_Organization,Establishment of protein localization
Intracellular_Organization,Gene expression
Intracellular_Organization,Histone H2A acetylation
Intracellular_Organization,Protein kinase C binding
Intracellular_Organization,Kinesin binding
Intracellular_Organization,Signaling receptor binding
Intracellular_Organization,Regulation of catalytic activity
Intracellular_Organization,Structural molecule activity
Intracellular_Organization,External side of plasma membrane
Intracellular_Organization,Hydrolase activity
Stress_Ion_Redox,Response to gamma radiation
Stress_Ion_Redox,Response to cold
Stress_Ion_Redox,Response to cadmium ion
Stress_Ion_Redox,Response to zinc ion
Stress_Ion_Redox,Response to hydrogen peroxide
Stress_Ion_Redox,Oxygen binding
Stress_Ion_Redox,Calcium ion binding
Stress_Ion_Redox,Positive regulation of release of sequestered calcium ion into cytosol
Stress_Ion_Redox,Regulation of mitochondrial membrane potential
Stress_Ion_Redox,Iron ion homeostasis
Stress_Ion_Redox,Intracellular iron ion homeostasis
Stress_Ion_Redox,Ferrous iron binding
Stress_Ion_Redox,Negative regulation of extrinsic apoptotic signaling pathway via death domain receptors"""

# Create DataFrame
go_terms_df = pd.read_csv(io.StringIO(go_terms_csv))

# Create dictionary (category -> list of GO terms)
go_terms_dict = go_terms_df.groupby('Category')['GO_Term'].apply(list).to_dict()

print("DataFrame shape:", go_terms_df.shape)
print("\nCategories:", list(go_terms_dict.keys()))
print("\nNumber of terms per category:")
for cat, terms in go_terms_dict.items():
    print(f"  {cat}: {len(terms)} terms")

In [ ]:
for cat in go_terms_df["Category"]:
    print(cat)

In [ ]:
# top_proj40.index = [idx.capitalize() for idx in top_proj40.index]
# top_proj40.index = top_proj40.index.str.replace(' t ', ' T ')
# top_proj40.index = top_proj40.index.str.replace(' i ', ' I ')
# top_proj40.index = top_proj40.index.str.replace(' ii ', ' II ')

# top_proj40.index = top_proj40.index.str.replace('Er ', 'ER ')
# top_proj40.index = top_proj40.index.str.replace(' ca1 ', ' CA1 ')
# top_proj40.index = top_proj40.index.str.replace(' golgi ', ' Golgi ')
# top_proj40.index = top_proj40.index.str.replace(' c ', ' C ')
# top_proj40.index = top_proj40.index.str.replace('Swi/snf ', 'SWI/SNF ')
# top_proj40.index = top_proj40.index.str.replace('h2a ', 'H2A ')
top_proj40.index = [idx[0].upper() + idx[1:] if len(idx) > 0 else idx for idx in top_proj40.index]


In [ ]:
# Create boxplots for each category
from scipy import stats
from statannotations.Annotator import Annotator

top_proj_40_ann = top_proj40.merge(go_terms_df, left_index=True, right_on='GO_Term', how='left')
categories = top_proj_40_ann['Category'].dropna().unique()

conditions_order = ["ctrl_basal", "vcp_basal", "ctrl_A1 low", "vcp_A1 low", "ctrl_A1 high", "vcp_A1 high"]
plt.figure(figsize=(20, 6))
for i, category in enumerate(categories):
    subset = top_proj_40_ann[top_proj_40_ann['Category'] == category]
    
    # Melt the data for seaborn boxplot
    data_melted = subset[conditions_order].melt(
        var_name='Condition', value_name='Value'
    )
    
    # Define pairs for statistical comparison
    pairs = [('ctrl_basal', 'vcp_basal'), 
             ('ctrl_A1 low', 'vcp_A1 low'), 
             ('ctrl_A1 high', 'vcp_A1 high')]
    
    plt.subplot(1, 6, i+1)
    ax = sns.boxplot(data=data_melted, x='Condition', y='Value', 
                     palette=palette_treatments, order=conditions_order)
    
    # Calculate p-values manually and format them
    formatted_pvalues = []
    for pair in pairs:
        group1 = data_melted[data_melted['Condition'] == pair[0]]['Value']
        group2 = data_melted[data_melted['Condition'] == pair[1]]['Value']
        _, pval = stats.mannwhitneyu(group1, group2, alternative='two-sided')
        formatted_pvalues.append(f'P={pval:.2e}')
    
    # Add statistical annotations with p-values
    annotator = Annotator(ax, pairs, data=data_melted, x='Condition', y='Value', order=conditions_order)
    annotator.configure(fontsize=10, loc='inside')
    annotator.set_custom_annotations(formatted_pvalues)
    annotator.annotate()
    
    plt.xticks(rotation=45, ha='right')
    plt.title(f'{category} \n ({subset.shape[0]} pathways)', weight='bold', fontsize=14)
    plt.ylabel('Signed -log10(p-value)', fontsize=12)
    plt.xlabel('')
    sns.despine()
    plt.ylim(-14, 14)
    plt.axhline(0, color='gray', linestyle='dashed', linewidth=1)
plt.tight_layout()

plt.savefig(os.path.join(path_to_figures, folder_figures, f"boxplot_treatment.pdf"), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
go_terms = pd.read_csv("/path/to/annotation/GO/human/human_GO_terms.csv")
go_terms['Description'] = [idx[0].upper() + idx[1:] if len(idx) > 0 else idx for idx in go_terms['Description'].values]

top_proj_40_ann = top_proj_40_ann.merge(go_terms, left_on="GO_Term", right_on="Description")

In [ ]:
pathways_all = {}
for category in top_proj_40_ann["Category"].unique():
    pathways_all[category] = {}
    print(f"######### {category} #########")
    for idx, row in top_proj_40_ann[top_proj_40_ann["Category"] == category].iterrows():
        pathways_all[category][row["GO_Term"]] = set(row["Gene_Symbols"].split(', '))

In [ ]:
import itertools
import networkx as nx
import matplotlib.pyplot as plt

category = "Cytoskeleton_Adhesion_Motility"
pathways = pathways_all[category]

# Function to compute Jaccard index
def jaccard(set1, set2):
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union > 0 else 0

# Build graph
G = nx.Graph()

# Create mappings from GO_Term to values
pathway_values_ctrl = {}
pathway_values_vcp = {}
for pathway in pathways:
    # Find the value in top_proj_40_ann
    match = top_proj_40_ann[top_proj_40_ann['GO_Term'] == pathway]
    if not match.empty:
        pathway_values_ctrl[pathway] = match['ctrl_basal'].values[0]
        pathway_values_vcp[pathway] = match['vcp_basal'].values[0]
    else:
        pathway_values_ctrl[pathway] = 0  # Default if not found
        pathway_values_vcp[pathway] = 0

# Add nodes
for pathway in pathways:
    G.add_node(pathway, size=len(pathways[pathway]), 
               ctrl_value=pathway_values_ctrl[pathway],
               vcp_value=pathway_values_vcp[pathway])

# Compute edges
threshold = 0.01  # Adjust this

for (p1, genes1), (p2, genes2) in itertools.combinations(pathways.items(), 2):
    score = jaccard(genes1, genes2)
    if score >= threshold:
        G.add_edge(p1, p2, weight=score)

print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")

# Calculate layout once for consistency across subplots
pos = nx.kamada_kawai_layout(G)

sizes = [G.nodes[n]['size'] * 50 for n in G.nodes]

# Get edge weights for coloring
edges = G.edges()
weights = [G[u][v]['weight'] for u, v in edges]

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(32, 12))

# Subplot 1: ctrl_basal
node_colors_ctrl = [G.nodes[n]['ctrl_value'] for n in G.nodes]

ax1.set_title(f"{category}\nctrl_basal enrichment", 
              fontsize=16, weight='bold', pad=20)

# Draw edges
nx.draw_networkx_edges(G, pos, width=[w*40 for w in weights], 
                        alpha=1, edge_color=weights, 
                        edge_cmap=plt.cm.Blues, edge_vmin=0, edge_vmax=0.2,
                        ax=ax1)

# Draw nodes colored by ctrl_basal value
nodes1 = nx.draw_networkx_nodes(G, pos, node_size=sizes, 
                       node_color=node_colors_ctrl, 
                       cmap=cmap,
                       vmin=-14, vmax=14,
                       edgecolors='black', linewidths=2,
                       ax=ax1)

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax1)

# Add colorbar
cbar1 = plt.colorbar(nodes1, ax=ax1, label='Signed -log10(p-value)', shrink=0.8)

ax1.axis('off')

# Subplot 2: vcp_basal
node_colors_vcp = [G.nodes[n]['vcp_value'] for n in G.nodes]

ax2.set_title(f"{category}\nvcp_basal enrichment", 
              fontsize=16, weight='bold', pad=20)

# Draw edges
nx.draw_networkx_edges(G, pos, width=[w*40 for w in weights], 
                        alpha=1, edge_color=weights, 
                        edge_cmap=plt.cm.Blues, edge_vmin=0, edge_vmax=0.2,
                        ax=ax2)

# Draw nodes colored by vcp_basal value
nodes2 = nx.draw_networkx_nodes(G, pos, node_size=sizes, 
                       node_color=node_colors_vcp, 
                       cmap=cmap,
                       vmin=-14, vmax=14,
                       edgecolors='black', linewidths=2,
                       ax=ax2)

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax2)

# Add colorbar
cbar2 = plt.colorbar(nodes2, ax=ax2, label='Signed -log10(p-value)', shrink=0.8)

ax2.axis('off')

plt.suptitle(f"Pathway Overlap Network (Jaccard ≥ {threshold})\n{G.number_of_nodes()} pathways, {G.number_of_edges()} connections", 
          fontsize=18, weight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
def plot_pathway_network(pathways, top_proj_40_ann, conditions=None, category_name=None,
                         threshold=0.01, colormap=cmap, vmin=-14, vmax=14, 
                         figsize_per_subplot=(16, 12), node_size_multiplier=50, 
                         edge_width_multiplier=40, font_size=8, default_node_color='lightblue',
                         layout='kamada_kawai', layout_kwargs=None):
    """
    Create a network visualization of pathway overlap with multiple subplots colored by different conditions.
    
    Parameters:
    -----------
    pathways : dict
        Dictionary where keys are pathway names and values are sets of genes
    top_proj_40_ann : pd.DataFrame
        DataFrame containing 'GO_Term' column and condition columns with enrichment values
    conditions : list, optional
        List of condition column names to visualize (e.g., ['ctrl_basal', 'vcp_basal'])
        If None, creates a single plot with uniformly colored nodes
    category_name : str, optional
        Name of the category for the plot title. If None, uses generic title
    threshold : float, default=0.01
        Jaccard similarity threshold for creating edges between pathways
    colormap : matplotlib colormap, default=cmap
        Colormap to use for node colors (only used when conditions are provided)
    vmin, vmax : float, default=-14, 14
        Color scale limits for node colors (only used when conditions are provided)
    figsize_per_subplot : tuple, default=(16, 12)
        Figure size per subplot
    node_size_multiplier : int, default=50
        Multiplier for node sizes (size = num_genes * multiplier)
    edge_width_multiplier : int, default=40
        Multiplier for edge widths (width = jaccard_score * multiplier)
    font_size : int, default=8
        Font size for pathway labels
    default_node_color : str, default='lightblue'
        Node color when no conditions are specified
    layout : str, default='kamada_kawai'
        Layout algorithm to use. Options: 'kamada_kawai', 'spring', 'circular', 'spectral', 'shell'
        Use different layouts if nodes are overlapping
    layout_kwargs : dict, optional
        Additional keyword arguments to pass to the layout function.
        Common options:
        - For 'spring': k (optimal distance), iterations (default 50), seed
        - For 'kamada_kawai': scale (default 1), seed
        Example: layout_kwargs={'scale': 3, 'seed': 42} or {'k': 0.5, 'iterations': 100}
        
    Returns:
    --------
    fig, axes : matplotlib figure and axes
    G : networkx Graph
    """
    import itertools
    import networkx as nx
    import matplotlib.pyplot as plt
    
    # Handle None or empty conditions
    if conditions is None:
        conditions = []
    
    # Function to compute Jaccard index
    def jaccard(set1, set2):
        intersection = len(set1 & set2)
        union = len(set1 | set2)
        return intersection / union if union > 0 else 0
    
    # Build graph
    G = nx.Graph()
    
    # Create mappings from GO_Term to values for each condition
    if conditions:
        pathway_values = {cond: {} for cond in conditions}
        
        for pathway in pathways:
            # Find the value in top_proj_40_ann
            match = top_proj_40_ann[top_proj_40_ann['GO_Term'] == pathway]
            
            for cond in conditions:
                if not match.empty and cond in match.columns:
                    pathway_values[cond][pathway] = match[cond].values[0]
                else:
                    pathway_values[cond][pathway] = 0  # Default if not found
        
        # Add nodes with attributes for each condition
        for pathway in pathways:
            node_attrs = {'size': len(pathways[pathway])}
            for cond in conditions:
                node_attrs[f'{cond}_value'] = pathway_values[cond][pathway]
            G.add_node(pathway, **node_attrs)
    else:
        # Add nodes without condition values
        for pathway in pathways:
            G.add_node(pathway, size=len(pathways[pathway]))
    
    # Compute edges based on Jaccard similarity
    for (p1, genes1), (p2, genes2) in itertools.combinations(pathways.items(), 2):
        score = jaccard(genes1, genes2)
        if score >= threshold:
            G.add_edge(p1, p2, weight=score)
    
    print(f"Number of nodes: {G.number_of_nodes()}")
    print(f"Number of edges: {G.number_of_edges()}")
    
    # Set default layout kwargs if not provided
    if layout_kwargs is None:
        layout_kwargs = {}
    
    # Calculate layout once for consistency across subplots
    if layout == 'kamada_kawai':
        # Set default scale if not provided
        if 'scale' not in layout_kwargs:
            layout_kwargs['scale'] = 2
        pos = nx.kamada_kawai_layout(G, **layout_kwargs)
    elif layout == 'spring':
        # Set default parameters if not provided
        if 'k' not in layout_kwargs:
            layout_kwargs['k'] = None  # Will auto-calculate optimal distance
        if 'iterations' not in layout_kwargs:
            layout_kwargs['iterations'] = 50
        pos = nx.spring_layout(G, **layout_kwargs)
    elif layout == 'circular':
        pos = nx.circular_layout(G, **layout_kwargs)
    elif layout == 'spectral':
        pos = nx.spectral_layout(G, **layout_kwargs)
    elif layout == 'shell':
        pos = nx.shell_layout(G, **layout_kwargs)
    else:
        raise ValueError(f"Unknown layout: {layout}. Choose from 'kamada_kawai', 'spring', 'circular', 'spectral', 'shell'")
    
    sizes = [G.nodes[n]['size'] * node_size_multiplier for n in G.nodes]
    
    # Get edge weights for coloring
    edges = G.edges()
    weights = [G[u][v]['weight'] for u, v in edges]
    
    # Create figure with subplots
    if conditions:
        n_conditions = len(conditions)
        fig, axes = plt.subplots(1, n_conditions, 
                                 figsize=(figsize_per_subplot[0] * n_conditions, figsize_per_subplot[1]))
        
        # Ensure axes is always a list
        if n_conditions == 1:
            axes = [axes]
        
        # Create a subplot for each condition
        for idx, (cond, ax) in enumerate(zip(conditions, axes)):
            node_colors = [G.nodes[n][f'{cond}_value'] for n in G.nodes]
            
            ax.set_title(f"{cond}", 
                         fontsize=16, weight='bold', pad=20)
            
            # Draw edges
            nx.draw_networkx_edges(G, pos, width=[w * edge_width_multiplier for w in weights], 
                                  alpha=1, edge_color=weights, 
                                  edge_cmap=plt.cm.Greys, edge_vmin=0, edge_vmax=0.2,
                                  ax=ax)
            
            # Draw nodes colored by condition value
            nodes = nx.draw_networkx_nodes(G, pos, node_size=sizes, 
                                           node_color=node_colors, 
                                           cmap=colormap,
                                           vmin=vmin, vmax=vmax,
                                           edgecolors='black', linewidths=2,
                                           ax=ax)
            
            # Draw labels
            nx.draw_networkx_labels(G, pos, font_size=font_size, font_weight='bold', ax=ax)
            
            # Add colorbar
            plt.colorbar(nodes, ax=ax, label='Signed -log10(p-value)', shrink=0.8)
            
            ax.axis('off')
    else:
        # Single plot with uniform node colors
        fig, ax = plt.subplots(1, 1, figsize=figsize_per_subplot)
        axes = [ax]
        
        ax.set_title(f"{category_name if category_name else 'Pathway Overlap Network'}", 
                     fontsize=16, weight='bold', pad=20)
        
        # Draw edges
        nx.draw_networkx_edges(G, pos, width=[w * edge_width_multiplier for w in weights], 
                              alpha=1, edge_color=weights, 
                              edge_cmap=plt.cm.Greys, edge_vmin=0, edge_vmax=0.2,
                              ax=ax)
        
        # Draw nodes with uniform color
        nodes = nx.draw_networkx_nodes(G, pos, node_size=sizes, 
                                       node_color=default_node_color, 
                                       edgecolors='black', linewidths=2,
                                       ax=ax)
        
        # Draw labels
        nx.draw_networkx_labels(G, pos, font_size=font_size, font_weight='bold', ax=ax)
        
        ax.axis('off')
    
    # Add overall title
    if conditions:
        title = f"{category_name if category_name else 'Pathways'}"
        plt.suptitle(title, fontsize=18, weight='bold', y=0.98)
    
    plt.tight_layout()
    
    return fig, axes, G

In [ ]:
len(list(pathways_all["Cytoskeleton_Adhesion_Motility"].keys()))

In [ ]:
# Example usage: Create network visualization for Cytoskeleton_Adhesion_Motility category
# comparing ctrl_basal and vcp_basal

# conditions_to_plot = ['ctrl_basal', 'ctrl_A1 low', 'ctrl_A1 high']

for category in top_proj_40_ann["Category"].unique():
    for layout in ["circular"]:
        for conditions_to_plot in [['ctrl_basal', 'ctrl_A1 low', 'ctrl_A1 high'],
                                ['vcp_basal', 'vcp_A1 low', 'vcp_A1 high']]:
            pathways = pathways_all[category]

            fig, axes, G = plot_pathway_network(
                pathways=pathways,
                top_proj_40_ann=top_proj_40_ann,
                conditions=conditions_to_plot,
                category_name=category,
                threshold=0.01,
                font_size=13,
                colormap=cmap,
                vmin=-14,
                vmax=14,
                layout=layout,
                edge_width_multiplier=5,
                
            )
            if conditions_to_plot is None:
                conditions_to_plot = ""
            plt.savefig(os.path.join(path_to_figures, 
                                    folder_figures, 
                                    f"network_{category}_{'_'.join(conditions_to_plot)}_{layout}.pdf"), 
                        bbox_inches='tight', 
                        dpi=300)


In [ ]:
# Example usage: Create network visualization for Cytoskeleton_Adhesion_Motility category
# comparing ctrl_basal and vcp_basal

# conditions_to_plot = ['ctrl_basal', 'ctrl_A1 low', 'ctrl_A1 high']

for category in top_proj_40_ann["Category"].unique():
    for layout in ["circular"]:
        for conditions_to_plot in [None]:
            pathways = pathways_all[category]

            fig, axes, G = plot_pathway_network(
                pathways=pathways,
                top_proj_40_ann=top_proj_40_ann,
                conditions=conditions_to_plot,
                category_name=category,
                threshold=0.01,
                font_size=13,
                colormap=cmap,
                vmin=-14,
                vmax=14,
                layout=layout,
                edge_width_multiplier=5
                
            )
            if conditions_to_plot is None:
                conditions_to_plot = ""
            plt.savefig(os.path.join(path_to_figures, 
                                    folder_figures, 
                                    f"network_{category}_{'_'.join(conditions_to_plot)}_{layout}.pdf"), 
                        bbox_inches='tight', 
                        dpi=300)


In [ ]:
projection_PC3 = pvalues_dataset.Vt_w1[2]
indexes_top = np.argsort(projection_PC3)[-50:]
indexes_bottom = np.argsort(projection_PC3)[:50]

top_highest_PC3 = list(pvalues_dataset.df.reset_index().iloc[indexes_top]['index'].values)
top_lowest_PC3 = list(pvalues_dataset.df.reset_index().iloc[indexes_bottom]['index'].values)

In [ ]:
top_proj40 = pd.concat([
              pvalues_dataset.X_without_PC1.T.loc[top_highest_PC3],
              pvalues_dataset.X_without_PC1.T.loc[top_lowest_PC3],
              ]).drop_duplicates(keep='first')




In [ ]:
palette_genotype = {"ctrl_basal": 'palegreen', "ctrl_A1 low": "limegreen", "ctrl_A1 high": "forestgreen", "vcp_basal": "pink", "vcp_A1 low": "hotpink", "vcp_A1 high": "deeppink" }

# palette_genotype = {"ctrl_basal": 'limegreen', "ctrl_A1 low": "limegreen", "ctrl_A1 high": "limegreen", "vcp_basal": "hotpink", "vcp_A1 low": "hotpink", "vcp_A1 high": "hotpink" }



In [ ]:

# Update col_colors to match remaining columns
col_colors_clean = [palette_genotype[treatment] for treatment in top_proj40.columns]
            
            
# First try with standard scaling
g = sns.clustermap(top_proj40, 
                    col_colors=col_colors_clean,
                    col_cluster=True, 
                    row_cluster=True, 
                    yticklabels=True, 
    cmap=cmap,
    center=0,  
                    figsize=(12, 20))


g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(), 
    rotation=45, 
    ha='right'
)
plt.savefig(os.path.join(path_to_figures, folder_figures, "BP_clustermap_top50_PC3_removed.pdf"), bbox_inches='tight', dpi=300)

In [ ]:

go_terms_csv = """Category,GO_Term
Cytoskeleton_Adhesion_Motility,Cell-cell junction
Cytoskeleton_Adhesion_Motility,Tight junction
Cytoskeleton_Adhesion_Motility,Axonal growth cone
Cytoskeleton_Adhesion_Motility,Positive regulation of fibroblast proliferation
Cytoskeleton_Adhesion_Motility,Positive regulation of cell migration
Cytoskeleton_Adhesion_Motility,Negative regulation of vascular endothelial cell proliferation
Cytoskeleton_Adhesion_Motility,Negative regulation of fibroblast proliferation
Cytoskeleton_Adhesion_Motility,Regulation of G0 to G1 transition
Cytoskeleton_Adhesion_Motility,Positive regulation of axon extension
Cytoskeleton_Adhesion_Motility,Regulation of cell morphogenesis
Cytoskeleton_Adhesion_Motility,Regulation of cell migration
Cytoskeleton_Adhesion_Motility,Myoblast fusion
Cytoskeleton_Adhesion_Motility,Positive regulation of vascular associated smooth muscle cell migration
Cytoskeleton_Adhesion_Motility,Z disc
Cytoskeleton_Adhesion_Motility,Plasma membrane repair
Cytoskeleton_Adhesion_Motility,Cell-cell junction organization
Cytoskeleton_Adhesion_Motility,Stress fiber
Cytoskeleton_Adhesion_Motility,Negative regulation of myoblast differentiation
Cytoskeleton_Adhesion_Motility,Osteoblast differentiation
Cytoskeleton_Adhesion_Motility,Myosin II complex
Cytoskeleton_Adhesion_Motility,Actin filament
Cytoskeleton_Adhesion_Motility,Negative regulation of cell population proliferation

ECM_Tissue_Remodeling,Extracellular matrix binding
ECM_Tissue_Remodeling,Peptide cross-linking
ECM_Tissue_Remodeling,Positive regulation of wound healing
ECM_Tissue_Remodeling,Positive regulation of collagen biosynthetic process
ECM_Tissue_Remodeling,Negative regulation of angiogenesis
ECM_Tissue_Remodeling,Blood circulation
ECM_Tissue_Remodeling,Heart morphogenesis
ECM_Tissue_Remodeling,Fibroblast growth factor binding
ECM_Tissue_Remodeling,Heparin binding
ECM_Tissue_Remodeling,Megakaryocyte development

Neuronal_Synaptic,Schaffer collateral - CA1 synapse
Neuronal_Synaptic,Response to axon injury
Neuronal_Synaptic,Neuron projection development
Neuronal_Synaptic,Long-term synaptic potentiation
Neuronal_Synaptic,Semaphorin-plexin signaling pathway
Neuronal_Synaptic,Cell body
Neuronal_Synaptic,Neuromuscular junction development
Neuronal_Synaptic,Neuromuscular junction

Immune_Antiviral,MHC class II protein complex binding
Immune_Antiviral,Positive regulation of chemokine production
Immune_Antiviral,Killing of cells of another organism
Immune_Antiviral,Tertiary granule lumen
Immune_Antiviral,Positive regulation of nitric oxide biosynthetic process
Immune_Antiviral,Peptide antigen binding
Immune_Antiviral,Phagocytic vesicle membrane
Immune_Antiviral,Positive regulation of T cell differentiation
Immune_Antiviral,Positive regulation of T cell mediated cytotoxicity
Immune_Antiviral,Superoxide anion generation
Immune_Antiviral,Immune response
Immune_Antiviral,Positive regulation of T cell activation
Immune_Antiviral,Antigen processing and presentation of exogenous peptide antigen via MHC class II
Immune_Antiviral,T cell activation involved in immune response
Immune_Antiviral,Microglial cell activation
Immune_Antiviral,Chemoattractant activity
Immune_Antiviral,Positive regulation of interferon-alpha production
Immune_Antiviral,Positive regulation of monocyte chemotaxis
Immune_Antiviral,Antibacterial humoral response
Immune_Antiviral,Negative regulation of viral entry into host cell
Immune_Antiviral,Chemokine activity
Immune_Antiviral,Positive regulation of interferon-beta production
Immune_Antiviral,Specific granule lumen

Intracellular_Organization,Negative regulation of protein binding
Intracellular_Organization,Regulation of nucleotide-excision repair
Intracellular_Organization,NuA4 histone acetyltransferase complex
Intracellular_Organization,Positive regulation of receptor-mediated endocytosis
Intracellular_Organization,Structural constituent of ribosome
Intracellular_Organization,Lumenal side of endoplasmic reticulum membrane
Intracellular_Organization,GTPase binding
Intracellular_Organization,Kinesin binding
Intracellular_Organization,Lysosomal transport
Intracellular_Organization,Low-density lipoprotein particle receptor binding
Intracellular_Organization,Regulation of ERK1 and ERK2 cascade
Intracellular_Organization,Histone H2A acetylation
Intracellular_Organization,Nucleosomal DNA binding
Intracellular_Organization,SMAD protein signal transduction
Intracellular_Organization,Chaperone binding
Intracellular_Organization,ER to Golgi transport vesicle membrane
Intracellular_Organization,Negative regulation of release of cytochrome c from mitochondria
Intracellular_Organization,Polysomal ribosome
Intracellular_Organization,Protein-membrane adaptor activity
Intracellular_Organization,Response to cAMP
Intracellular_Organization,Mitotic spindle assembly
Intracellular_Organization,Establishment of protein localization
Intracellular_Organization,Gene expression
Intracellular_Organization,Perinuclear region of cytoplasm
Intracellular_Organization,Transmembrane transporter binding
Intracellular_Organization,Cellular response to cAMP
Intracellular_Organization,Protein homotetramerization
Intracellular_Organization,Positive regulation of protein kinase B signaling
Intracellular_Organization,Protein homodimerization activity
Intracellular_Organization,Misfolded protein binding

Stress_Ion_Redox,Negative regulation of amyloid-beta formation
Stress_Ion_Redox,Oxygen binding
Stress_Ion_Redox,Positive regulation of release of sequestered calcium ion into cytosol
Stress_Ion_Redox,Intracellular iron ion homeostasis
Stress_Ion_Redox,Aging"""




In [ ]:

go_terms_csv = """Category,GO_Term
Cytoskeletal_Dynamics_Remodeling,Cell-cell junction
Cytoskeletal_Dynamics_Remodeling,Tight junction
Cytoskeletal_Dynamics_Remodeling,Axonal growth cone
Cytoskeletal_Dynamics_Remodeling,Cell projection
Cytoskeletal_Dynamics_Remodeling,Stress fiber
Cytoskeletal_Dynamics_Remodeling,Actin filament
Cytoskeletal_Dynamics_Remodeling,Z disc
Cytoskeletal_Dynamics_Remodeling,Myosin II complex
Cytoskeletal_Dynamics_Remodeling,Cell-cell junction organization
Cytoskeletal_Dynamics_Remodeling,Plasma membrane repair
Cytoskeletal_Dynamics_Remodeling,Myoblast fusion
Cytoskeletal_Dynamics_Remodeling,Neuromuscular junction
Cytoskeletal_Dynamics_Remodeling,Neuromuscular junction development
Cytoskeletal_Dynamics_Remodeling,Neuron projection development
Cytoskeletal_Dynamics_Remodeling,Positive regulation of axon extension
Cytoskeletal_Dynamics_Remodeling,Regulation of cell morphogenesis
Cytoskeletal_Dynamics_Remodeling,Positive regulation of cell migration
Cytoskeletal_Dynamics_Remodeling,Regulation of cell migration
Cytoskeletal_Dynamics_Remodeling,Positive regulation of vascular associated smooth muscle cell migration

Immune_Effector_Programs,MHC class II protein complex binding
Immune_Effector_Programs,Antigen processing and presentation of exogenous peptide antigen via MHC class II
Immune_Effector_Programs,Peptide antigen binding
Immune_Effector_Programs,Immune response
Immune_Effector_Programs,T cell activation involved in immune response
Immune_Effector_Programs,Positive regulation of T cell activation
Immune_Effector_Programs,Positive regulation of T cell differentiation
Immune_Effector_Programs,Positive regulation of T cell mediated cytotoxicity
Immune_Effector_Programs,Positive regulation of chemokine production
Immune_Effector_Programs,Chemokine activity
Immune_Effector_Programs,Chemoattractant activity
Immune_Effector_Programs,Positive regulation of monocyte chemotaxis
Immune_Effector_Programs,Killing of cells of another organism
Immune_Effector_Programs,Antibacterial humoral response
Immune_Effector_Programs,Microglial cell activation
Immune_Effector_Programs,Tertiary granule lumen
Immune_Effector_Programs,Specific granule lumen
Immune_Effector_Programs,Phagocytic vesicle membrane
Immune_Effector_Programs,Superoxide anion generation
Immune_Effector_Programs,Positive regulation of nitric oxide biosynthetic process
Immune_Effector_Programs,Positive regulation of interferon-alpha production
Immune_Effector_Programs,Positive regulation of interferon-beta production
Immune_Effector_Programs,Negative regulation of viral entry into host cell

ECM_GrowthFactor_TissueRepair,Extracellular matrix binding
ECM_GrowthFactor_TissueRepair,Peptide cross-linking
ECM_GrowthFactor_TissueRepair,Positive regulation of wound healing
ECM_GrowthFactor_TissueRepair,Positive regulation of collagen biosynthetic process
ECM_GrowthFactor_TissueRepair,Negative regulation of angiogenesis
ECM_GrowthFactor_TissueRepair,Blood circulation
ECM_GrowthFactor_TissueRepair,Heart morphogenesis
ECM_GrowthFactor_TissueRepair,Fibroblast growth factor binding
ECM_GrowthFactor_TissueRepair,Heparin binding
ECM_GrowthFactor_TissueRepair,Positive regulation of fibroblast proliferation
ECM_GrowthFactor_TissueRepair,Negative regulation of fibroblast proliferation
ECM_GrowthFactor_TissueRepair,Negative regulation of vascular endothelial cell proliferation
ECM_GrowthFactor_TissueRepair,Osteoblast differentiation
ECM_GrowthFactor_TissueRepair,Megakaryocyte development

Mitogenic_Differentiation_Signaling,Regulation of ERK1 and ERK2 cascade
Mitogenic_Differentiation_Signaling,SMAD protein signal transduction
Mitogenic_Differentiation_Signaling,Positive regulation of protein kinase B signaling
Mitogenic_Differentiation_Signaling,Response to cAMP
Mitogenic_Differentiation_Signaling,Cellular response to cAMP
Mitogenic_Differentiation_Signaling,Positive regulation of receptor-mediated endocytosis
Mitogenic_Differentiation_Signaling,Transmembrane transporter binding
Mitogenic_Differentiation_Signaling,GTPase binding
Mitogenic_Differentiation_Signaling,Kinesin binding
Mitogenic_Differentiation_Signaling,Protein-membrane adaptor activity
Mitogenic_Differentiation_Signaling,Low-density lipoprotein particle receptor binding
Mitogenic_Differentiation_Signaling,Semaphorin-plexin signaling pathway

Chromatin_DNARepair_Translation,NuA4 histone acetyltransferase complex
Chromatin_DNARepair_Translation,Histone H2A acetylation
Chromatin_DNARepair_Translation,Nucleosomal DNA binding
Chromatin_DNARepair_Translation,Regulation of nucleotide-excision repair
Chromatin_DNARepair_Translation,Gene expression
Chromatin_DNARepair_Translation,Structural constituent of ribosome
Chromatin_DNARepair_Translation,Polysomal ribosome
Chromatin_DNARepair_Translation,Protein homodimerization activity
Chromatin_DNARepair_Translation,Protein homotetramerization
Chromatin_DNARepair_Translation,Negative regulation of protein binding

Cell_Stress_Apoptosis_Proteostasis,Negative regulation of release of cytochrome c from mitochondria
Cell_Stress_Apoptosis_Proteostasis,Negative regulation of extrinsic apoptotic signaling pathway via death domain receptors
Cell_Stress_Apoptosis_Proteostasis,Negative regulation of amyloid-beta formation
Cell_Stress_Apoptosis_Proteostasis,Oxygen binding
Cell_Stress_Apoptosis_Proteostasis,Positive regulation of release of sequestered calcium ion into cytosol
Cell_Stress_Apoptosis_Proteostasis,Intracellular iron ion homeostasis
Cell_Stress_Apoptosis_Proteostasis,Aging
Cell_Stress_Apoptosis_Proteostasis,Lysosomal transport
Cell_Stress_Apoptosis_Proteostasis,Perinuclear region of cytoplasm
Cell_Stress_Apoptosis_Proteostasis,ER to Golgi transport vesicle membrane
Cell_Stress_Apoptosis_Proteostasis,Lumenal side of endoplasmic reticulum membrane
Cell_Stress_Apoptosis_Proteostasis,Mitotic spindle assembly
Cell_Stress_Apoptosis_Proteostasis,Regulation of G0 to G1 transition
Cell_Stress_Apoptosis_Proteostasis,Misfolded protein binding
Cell_Stress_Apoptosis_Proteostasis,Chaperone binding
Cell_Stress_Apoptosis_Proteostasis,Establishment of protein localization
Cell_Stress_Apoptosis_Proteostasis,Schaffer collateral - CA1 synapse
Cell_Stress_Apoptosis_Proteostasis,Long-term synaptic potentiation
Cell_Stress_Apoptosis_Proteostasis,Response to axon injury
Cell_Stress_Apoptosis_Proteostasis,Cell body"""


In [ ]:
# Create DataFrame
go_terms_df = pd.read_csv(io.StringIO(go_terms_csv))

# Create dictionary (category -> list of GO terms)
go_terms_dict = go_terms_df.groupby('Category')['GO_Term'].apply(list).to_dict()

In [ ]:
top_proj40.index = [idx[0].upper() + idx[1:] if len(idx) > 0 else idx for idx in top_proj40.index]

In [ ]:
# Create boxplots for each category


top_proj_40_ann = top_proj40.merge(go_terms_df, left_index=True, right_on='GO_Term', how='left')
categories = top_proj_40_ann['Category'].dropna().unique()

conditions_order = ["ctrl_basal", "ctrl_A1 high", "ctrl_A1 low", "vcp_basal", "vcp_A1 low",  "vcp_A1 high"]
plt.figure(figsize=(20, 6))
for i, category in enumerate(categories):
    subset = top_proj_40_ann[top_proj_40_ann['Category'] == category]
    
    # Melt the data for seaborn boxplot
    data_melted = subset[conditions_order].melt(
        var_name='Condition', value_name='Value'
    )
    
    plt.subplot(1, 6, i+1)
    sns.boxplot(data=data_melted, x='Condition', y='Value', 
                palette=palette_genotype, order=conditions_order)
    plt.xticks(rotation=45, ha='right')
    plt.title(f'{category} \n ({subset.shape[0]} pathways)', weight='bold', fontsize=12)
    plt.ylabel('Signed -log10(p-value)', fontsize=12)
    plt.xlabel('')
    sns.despine()
    plt.ylim(-12, 12)
    plt.axhline(0, color='gray', linestyle='dashed', linewidth=1)
plt.tight_layout()

plt.savefig(os.path.join(path_to_figures, folder_figures, f"boxplot_genotype.pdf"), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
from scipy import stats
from statannotations.Annotator import Annotator

top_proj_40_ann = top_proj40.merge(go_terms_df, left_index=True, right_on='GO_Term', how='left')
categories = top_proj_40_ann['Category'].dropna().unique()

conditions_order = ["ctrl_basal",  "ctrl_A1 low", "ctrl_A1 high", "vcp_basal", "vcp_A1 low",  "vcp_A1 high"]

# Define pairs for statistical comparison (ctrl vs vcp for each condition)
pairs = [('ctrl_basal', 'vcp_basal'), 
         ('ctrl_A1 low', 'vcp_A1 low'), 
         ('ctrl_A1 high', 'vcp_A1 high')]

plt.figure(figsize=(20, 6))
for i, category in enumerate(categories):
    subset = top_proj_40_ann[top_proj_40_ann['Category'] == category]
    
    # Melt the data for seaborn boxplot
    data_melted = subset[conditions_order].melt(
        var_name='Condition', value_name='Value'
    )
    
    plt.subplot(1, 6, i+1)
    ax = sns.boxplot(data=data_melted, x='Condition', y='Value', 
                palette=palette_genotype, order=conditions_order)

    # Calculate p-values manually and format them
    formatted_pvalues = []
    for pair in pairs:
        group1 = data_melted[data_melted['Condition'] == pair[0]]['Value']
        group2 = data_melted[data_melted['Condition'] == pair[1]]['Value']
        _, pval = stats.mannwhitneyu(group1, group2, alternative='two-sided')
        formatted_pvalues.append(f'P={pval:.2e}')

    # Add statistical annotations with p-values
    annotator = Annotator(ax, pairs, data=data_melted, x='Condition', y='Value', order=conditions_order)
    annotator.configure(fontsize=10, loc='inside')
    annotator.set_custom_annotations(formatted_pvalues)
    annotator.annotate()

    plt.xticks(rotation=45, ha='right')
    plt.title(f'{category} \n ({subset.shape[0]} pathways)', weight='bold', fontsize=12)
    plt.ylabel('Signed -log10(p-value)', fontsize=12)
    plt.xlabel('')
    sns.despine()
    plt.ylim(-12, 12)
    plt.axhline(0, color='gray', linestyle='dashed', linewidth=1)
plt.tight_layout()

plt.savefig(os.path.join(path_to_figures, folder_figures, f"boxplot_genotype.pdf"), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
top_proj_40_ann = top_proj_40_ann.merge(go_terms, left_on="GO_Term", right_on="Description")

pathways_all = {}
for category in top_proj_40_ann["Category"].unique():
    pathways_all[category] = {}
    print(f"######### {category} #########")
    for idx, row in top_proj_40_ann[top_proj_40_ann["Category"] == category].iterrows():
        pathways_all[category][row["GO_Term"]] = set(row["Gene_Symbols"].split(', '))

In [ ]:

for category in top_proj_40_ann["Category"].unique():
    for layout in ["circular"]:
        for conditions_to_plot in [['ctrl_basal', 'ctrl_A1 low', 'ctrl_A1 high'],
                                ['vcp_basal', 'vcp_A1 low', 'vcp_A1 high']]:
            pathways = pathways_all[category]

            fig, axes, G = plot_pathway_network(
                pathways=pathways,
                top_proj_40_ann=top_proj_40_ann,
                conditions=conditions_to_plot,
                category_name=category,
                threshold=0.01,
                font_size=13,
                colormap=cmap,
                vmin=-14,
                vmax=14,
                layout=layout,
                edge_width_multiplier=5
            )
            if conditions_to_plot is None:
                conditions_to_plot = ""
            plt.savefig(os.path.join(path_to_figures, 
                                    folder_figures, 
                                    f"network_{category}_{'_'.join(conditions_to_plot)}_{layout}.pdf"), 
                        bbox_inches='tight', 
                        dpi=300)

In [ ]:
go_terms = pd.read_csv("/path/to/annotation/GO/human/human_GO_terms.csv")
go_terms['Description'] = [idx[0].upper() + idx[1:] if len(idx) > 0 else idx for idx in go_terms['Description'].values]

top_proj_40_ann = top_proj_40_ann.merge(go_terms, left_on="GO_Term", right_on="Description")

In [ ]:
top_proj_40_ann

In [ ]:
for category in top_proj_40_ann["Category"].unique():
    for layout in ["circular"]:
        for conditions_to_plot in [None]:
            pathways = pathways_all[category]

            fig, axes, G = plot_pathway_network(
                pathways=pathways,
                top_proj_40_ann=top_proj_40_ann,
                conditions=conditions_to_plot,
                category_name=category,
                threshold=0.01,
                font_size=13,
                colormap=cmap,
                vmin=-14,
                vmax=14,
                layout=layout,
                edge_width_multiplier=5
                
            )
            if conditions_to_plot is None:
                conditions_to_plot = ""
            plt.savefig(os.path.join(path_to_figures, 
                                    folder_figures, 
                                    f"network_{category}_{'_'.join(conditions_to_plot)}_{layout}.pdf"), 
                        bbox_inches='tight', 
                        dpi=300)


In [ ]:
for category in top_proj_40_ann["Category"].unique():
    print(f"######### {category} #########")
    for go_id in top_proj_40_ann[top_proj_40_ann["Category"] == category]["GO_id"].values:
        print(go_id)